# CSIT 595 Assignment 1: Generative Trilemma

This self-contained notebook follows the assignment order and keeps all implementation code in modular sections. Run cells from top to bottom. Labels are used only for evaluator training and post-generation analysis; all generators are unconditional.

## Common Setup + Evaluator Network

This section implements preprocessing, the evaluator CNN, penultimate-layer feature extraction, cFID, coverage metrics, and sampling-cost utilities.

In [ ]:
import math
import random
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

CONFIG = {
    "full_run": False,
    "seed": 559,
    "batch_size": 128,
    "eval_epochs": 1,
    "vae_epochs": 1,
    "gan_epochs": 1,
    "diffusion_steps": 500,
    "diffusion_train_steps": 300,
    "metric_count": 500,
    "sample_count": 100,
    "num_workers": 0,
}


def configure_run(full_run=False):
    CONFIG["full_run"] = full_run
    if full_run:
        CONFIG.update({
            "eval_epochs": 5,
            "vae_epochs": 25,
            "gan_epochs": 25,
            "diffusion_train_steps": 3000,
            "metric_count": 2000,
        })
    else:
        CONFIG.update({
            "eval_epochs": 1,
            "vae_epochs": 1,
            "gan_epochs": 1,
            "diffusion_train_steps": 300,
            "metric_count": 500,
        })


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path.cwd()
OUT = ROOT / "assignment1_outputs"
OUT.mkdir(exist_ok=True)


def set_seed(seed: int = 559):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])


def make_loaders(batch_size=None, num_workers=None):
    bs = batch_size or CONFIG["batch_size"]
    workers = num_workers if num_workers is not None else CONFIG["num_workers"]
    transform = transforms.Compose([
        transforms.Pad(2),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    train_set = datasets.FashionMNIST(ROOT / "data", train=True, download=True, transform=transform)
    test_set = datasets.FashionMNIST(ROOT / "data", train=False, download=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=bs, shuffle=True, drop_last=True, num_workers=workers)
    test_loader = DataLoader(test_set, batch_size=bs, shuffle=False, num_workers=workers)
    return train_loader, test_loader


train_loader, test_loader = make_loaders()


class EvaluatorCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier_input = nn.Flatten()
        self.classifier_hidden = nn.Linear(128, 64)
        self.classifier_activation = nn.ReLU()
        self.classifier_output = nn.Linear(64, 10)

    def forward(self, x, return_features=False):
        feat = self.features(x)
        pooled = self.classifier_input(feat)
        hidden = self.classifier_activation(self.classifier_hidden(pooled))
        logits = self.classifier_output(hidden)
        if return_features:
            return logits, hidden
        return logits


def show_grid(images, title, filename, columns=10):
    images = ((images.detach().cpu().clamp(-1, 1) + 1) / 2).numpy()
    rows = max(1, math.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns, rows))
    axes = np.asarray(axes).reshape(-1)
    for idx, ax in enumerate(axes):
        ax.axis("off")
        if idx < len(images):
            ax.imshow(images[idx, 0], cmap="gray", vmin=0, vmax=1)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(OUT / filename, dpi=160)
    plt.close(fig)


def sqrt_psd(matrix):
    values, vectors = np.linalg.eigh((matrix + matrix.T) / 2)
    values = np.clip(values, 0, None)
    return (vectors * np.sqrt(values)) @ vectors.T


def coverage_metrics(real_features, real_labels, generated_features, generated_labels):
    real_mean = real_features.mean(0)
    gen_mean = generated_features.mean(0)
    real_cov = np.cov(real_features, rowvar=False) + 1e-6 * np.eye(real_features.shape[1])
    gen_cov = np.cov(generated_features, rowvar=False) + 1e-6 * np.eye(generated_features.shape[1])
    term = real_cov + gen_cov - 2 * sqrt_psd(real_cov @ gen_cov)
    cfid = np.sum((real_mean - gen_mean) ** 2) + np.trace(term)

    real_hist = np.bincount(real_labels, minlength=10).astype(np.float64)
    real_hist /= real_hist.sum()
    gen_hist = np.bincount(generated_labels, minlength=10).astype(np.float64)
    gen_hist /= gen_hist.sum()
    entropy = -np.sum(gen_hist * np.log(gen_hist + 1e-8))
    kl = np.sum(gen_hist * np.log((gen_hist + 1e-8) / (real_hist + 1e-8)))
    return {
        "cfid": float(cfid),
        "entropy": float(entropy),
        "kl_divergence": float(kl),
        "generated_histogram": gen_hist.tolist(),
    }


@torch.no_grad()
def evaluator_outputs(images, evaluator):
    logits, features = evaluator(images.to(DEVICE), return_features=True)
    return logits.argmax(1).cpu(), features.cpu()


def evaluator_accuracy(model):
    model.eval()
    correct = 0
    total = 0
    for images, labels in test_loader:
        logits = model(images.to(DEVICE))
        correct += (logits.argmax(1).cpu() == labels).sum().item()
        total += len(labels)
    return correct / total


def reference_features(count, evaluator):
    all_features, all_labels = [], []
    seen = 0
    for images, labels in test_loader:
        batch = min(len(images), count - seen)
        if batch <= 0:
            break
        _, features = evaluator_outputs(images[:batch], evaluator)
        all_features.append(features)
        all_labels.append(labels[:batch])
        seen += batch
    return torch.cat(all_features).numpy(), torch.cat(all_labels).numpy()


def predicted_histogram(images, evaluator):
    labels = []
    for start in range(0, len(images), CONFIG["batch_size"]):
        batch_labels, _ = evaluator_outputs(images[start:start + CONFIG["batch_size"]], evaluator)
        labels.append(batch_labels)
    hist = torch.bincount(torch.cat(labels), minlength=10).float()
    return hist / hist.sum()


def synchronize_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def train_evaluator(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    history = []
    model.train()
    for epoch in range(CONFIG["eval_epochs"]):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in tqdm(train_loader, desc=f"evaluator {epoch + 1}", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        history.append({"epoch": epoch + 1, "loss": running_loss / total, "acc": correct / total})
    return model, history


def evaluate_model(name, sample_fn, evaluator, training_seconds=None, parameters=None):
    count = CONFIG["metric_count"]
    synchronize_cuda()
    started = time.perf_counter()
    generated = sample_fn(count)
    synchronize_cuda()
    elapsed = time.perf_counter() - started
    labels, features = evaluator_outputs(generated, evaluator)
    real_features, real_labels = reference_features(count, evaluator)
    metrics = coverage_metrics(real_features, real_labels, features.numpy(), labels.numpy())
    metrics.update({
        "model": name,
        "images_per_second": count / elapsed,
        "time_for_5000_images": 5000 * elapsed / count,
        "parameters": parameters,
        "training_seconds": training_seconds,
    })
    return metrics, generated


# must be imported in notebooks after time module exists
import time


## 1. Variational Autoencoder (VAE)

Architecture, training objective, training curve, reconstruction, prior samples, and interpolation between two real test images.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm



class VAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 4, stride=2, padding=1),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(128 * 2 * 2, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 128 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),
            nn.Tanh(),
        )

    def encode(self, x):
        h = self.encoder(x).view(x.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_decode(z).view(-1, 128, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


def train_vae(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
    history = []
    started = __import__("time").perf_counter()
    for epoch in range(CONFIG["vae_epochs"]):
        model.train()
        recon_total = 0.0
        kl_total = 0.0
        loss_total = 0.0
        for images, _ in tqdm(train_loader, desc=f"VAE {epoch + 1}", leave=False):
            images = images.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            recon, mu, logvar = model(images)
            recon_loss = F.mse_loss(recon, images, reduction="mean")
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = recon_loss + 0.1 * kl
            loss.backward()
            optimizer.step()
            recon_total += recon_loss.item()
            kl_total += kl.item()
            loss_total += loss.item()
        history.append({
            "epoch": epoch + 1,
            "recon_loss": recon_total / len(train_loader),
            "kl": kl_total / len(train_loader),
            "total_loss": loss_total / len(train_loader),
        })
    return model, history, __import__("time").perf_counter() - started


@torch.no_grad()
def sample_vae(model, count):
    z = torch.randn(count, model.latent_dim, device=DEVICE)
    return model.decode(z).clamp(-1, 1)


## 2. Generative Adversarial Network (GAN)

DCGAN-style generator and discriminator, training losses, samples, interpolation, and checkpoint coverage analysis.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm



class Generator(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        z = z.view(z.size(0), z.size(1), 1, 1)
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0),
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)


def train_gan(generator, discriminator, checkpoint_callback=None):
    g_opt = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    d_opt = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    history = []
    started = __import__("time").perf_counter()
    for epoch in range(CONFIG["gan_epochs"]):
        generator.train()
        discriminator.train()
        g_loss_total = 0.0
        d_loss_total = 0.0
        for images, _ in tqdm(train_loader, desc=f"GAN {epoch + 1}", leave=False):
            images = images.to(DEVICE)
            batch = images.size(0)
            real_label = torch.ones(batch, device=DEVICE)
            fake_label = torch.zeros(batch, device=DEVICE)

            d_opt.zero_grad(set_to_none=True)
            real_logits = discriminator(images).view(-1)
            real_loss = F.binary_cross_entropy_with_logits(real_logits, real_label)
            noise = torch.randn(batch, generator.latent_dim, device=DEVICE)
            fake_images = generator(noise)
            fake_logits = discriminator(fake_images.detach()).view(-1)
            fake_loss = F.binary_cross_entropy_with_logits(fake_logits, fake_label)
            d_loss = real_loss + fake_loss
            d_loss.backward()
            d_opt.step()

            g_opt.zero_grad(set_to_none=True)
            noise = torch.randn(batch, generator.latent_dim, device=DEVICE)
            fake_images = generator(noise)
            fake_logits = discriminator(fake_images).view(-1)
            g_loss = F.binary_cross_entropy_with_logits(fake_logits, real_label)
            g_loss.backward()
            g_opt.step()

            g_loss_total += g_loss.item()
            d_loss_total += d_loss.item()
        history.append({"epoch": epoch + 1, "g_loss": g_loss_total / len(train_loader), "d_loss": d_loss_total / len(train_loader)})
        if checkpoint_callback is not None:
            checkpoint_callback(epoch + 1, generator)
    return generator, discriminator, history, __import__("time").perf_counter() - started


@torch.no_grad()
def sample_gan(generator, count):
    generator.eval()
    noise = torch.randn(count, generator.latent_dim, device=DEVICE)
    return generator(noise).clamp(-1, 1)


## 3. Diffusion Model

Forward noising process, timestep-conditioned U-Net, noise-prediction training, and 25/100/500-step sampling sweep.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm



class TinyUNet(nn.Module):
    def __init__(self, base_channels=32, time_dim=64):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(time_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
        )
        self.time_down1 = nn.Linear(128, base_channels * 2)
        self.time_down2 = nn.Linear(128, base_channels * 4)
        self.time_mid = nn.Linear(128, base_channels * 4)
        self.time_up1 = nn.Linear(128, base_channels * 2)
        self.time_up0 = nn.Linear(128, base_channels)
        self.inc = nn.Conv2d(1, base_channels, 3, padding=1)
        self.down1 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels * 4),
            nn.SiLU(),
        )
        self.mid = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 4, 3, padding=1),
            nn.GroupNorm(8, base_channels * 4),
            nn.SiLU(),
            nn.Conv2d(base_channels * 4, base_channels * 4, 3, padding=1),
            nn.GroupNorm(8, base_channels * 4),
            nn.SiLU(),
        )
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels * 2),
            nn.SiLU(),
        )
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
        )
        self.out = nn.Conv2d(base_channels, 1, 3, padding=1)

    def forward(self, x, timestep):
        timestep = timestep.to(device=x.device, dtype=x.dtype)
        frequencies = torch.arange(32, device=x.device, dtype=x.dtype)
        frequencies = torch.exp(-torch.log(torch.tensor(10000.0, device=x.device, dtype=x.dtype)) * frequencies / 31)
        angles = timestep[:, None] * frequencies[None, :]
        time_features = torch.cat((angles.sin(), angles.cos()), dim=1)
        cond = self.time_embed(time_features)
        def add_time(features, projection):
            return features + projection(cond).view(x.size(0), -1, 1, 1)

        x0 = self.inc(x)
        x1 = add_time(self.down1(x0), self.time_down1)
        x2 = add_time(self.down2(x1), self.time_down2)
        m = add_time(self.mid(x2), self.time_mid)
        u1 = add_time(self.up1(m), self.time_up1)
        u1 = u1 + x1
        u0 = add_time(self.up0(u1), self.time_up0)
        u0 = u0 + x0
        return self.out(u0)


class Diffusion:
    def __init__(self, steps=500):
        self.steps = steps
        self.betas = torch.linspace(1e-4, 0.02, steps, device=DEVICE)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, clean, t, noise=None):
        if noise is None:
            noise = torch.randn_like(clean)
        alpha_bar = self.alpha_bars[t].to(clean.device).view(-1, 1, 1, 1)
        return torch.sqrt(alpha_bar) * clean + torch.sqrt(1.0 - alpha_bar) * noise

    def sample(self, denoiser, count, steps=None):
        if steps is None:
            steps = self.steps
        denoiser.eval()
        x = torch.randn(count, 1, 32, 32, device=DEVICE)
        schedule = torch.linspace(self.steps - 1, 0, steps, device=DEVICE).round().long()
        for index, timestep in enumerate(schedule):
            t = torch.full((count,), timestep.item(), device=DEVICE, dtype=torch.long)
            with torch.no_grad():
                pred = denoiser(x, t)
            alpha_bar_t = self.alpha_bars[timestep]
            previous_timestep = schedule[index + 1] if index + 1 < len(schedule) else None
            alpha_bar_previous = torch.tensor(1.0, device=DEVICE) if previous_timestep is None else self.alpha_bars[previous_timestep]
            predicted_clean = (x - torch.sqrt(1.0 - alpha_bar_t) * pred) / torch.sqrt(alpha_bar_t)
            x = torch.sqrt(alpha_bar_previous) * predicted_clean + torch.sqrt(1.0 - alpha_bar_previous) * pred
        return x.clamp(-1, 1)


def train_diffusion(denoiser):
    optimizer = torch.optim.AdamW(denoiser.parameters(), lr=1e-4)
    history = []
    diffusion = Diffusion(steps=CONFIG["diffusion_steps"])
    started = __import__("time").perf_counter()
    denoiser.train()
    batches = iter(train_loader)
    for step in range(CONFIG["diffusion_train_steps"]):
        try:
            images, _ = next(batches)
        except StopIteration:
            batches = iter(train_loader)
            images, _ = next(batches)
        batch = images.to(DEVICE)
        t = torch.randint(0, diffusion.steps, (batch.size(0),), device=DEVICE)
        noise = torch.randn_like(batch)
        noisy = diffusion.add_noise(batch, t, noise=noise)
        pred = denoiser(noisy, t)
        loss = F.mse_loss(pred, noise)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % max(1, CONFIG["diffusion_train_steps"] // 30) == 0:
            history.append({"step": step + 1, "mse": loss.item()})
    return denoiser, diffusion, history, __import__("time").perf_counter() - started


@torch.no_grad()
def sample_diffusion(diffusion, denoiser, count, steps=None):
    return diffusion.sample(denoiser, count, steps=steps)


## Experiment Configuration

In [ ]:
# Full-run configuration. Set to False for a smoke test.
CONFIG["full_run"] = True
configure_run(CONFIG["full_run"])
print("Device:", DEVICE)
print("Full run:", CONFIG["full_run"])


## Run All Tasks in Assignment Order

This cell trains the evaluator, VAE, GAN, and diffusion model, evaluates cFID and coverage, generates all required visualizations, and saves CSV/PNG artifacts.

In [ ]:

def run_experiment():
    print("Training evaluator...")
    evaluator = EvaluatorCNN().to(DEVICE)
    evaluator, evaluator_history = train_evaluator(evaluator)
    print("Evaluator test accuracy:", evaluator_accuracy(evaluator))

    print("Training VAE...")
    vae = VAE(latent_dim=64).to(DEVICE)
    vae, vae_history, vae_seconds = train_vae(vae)

    print("Training GAN...")
    generator = Generator(latent_dim=128).to(DEVICE)
    discriminator = Discriminator().to(DEVICE)
    gan_checkpoints = []

    def record_gan_checkpoint(epoch, current_generator):
        current_generator.eval()
        with torch.no_grad():
            samples = current_generator(torch.randn(CONFIG["metric_count"], current_generator.latent_dim, device=DEVICE))
        histogram = predicted_histogram(samples, evaluator).tolist()
        gan_checkpoints.append({"epoch": epoch, **{f"class_{i}": value for i, value in enumerate(histogram)}})
        current_generator.train()

    generator, discriminator, gan_history, gan_seconds = train_gan(
        generator, discriminator, checkpoint_callback=record_gan_checkpoint
    )

    print("Training diffusion model...")
    denoiser = TinyUNet(base_channels=32, time_dim=64).to(DEVICE)
    denoiser, diffusion, diffusion_history, diffusion_seconds = train_diffusion(denoiser)

    vae_metrics, _ = evaluate_model(
        "VAE", lambda count: sample_vae(vae, count), evaluator,
        training_seconds=vae_seconds, parameters=sum(p.numel() for p in vae.parameters()),
    )
    generator_parameters = sum(p.numel() for p in generator.parameters())
    discriminator_parameters = sum(p.numel() for p in discriminator.parameters())
    gan_metrics, _ = evaluate_model(
        "GAN", lambda count: sample_gan(generator, count), evaluator,
        training_seconds=gan_seconds,
        parameters=generator_parameters + discriminator_parameters,
    )
    gan_metrics["generator_parameters"] = generator_parameters
    gan_metrics["discriminator_parameters"] = discriminator_parameters
    ddpm_metrics, _ = evaluate_model(
        "DDPM", lambda count: sample_diffusion(diffusion, denoiser, count, steps=CONFIG["diffusion_steps"]), evaluator,
        training_seconds=diffusion_seconds, parameters=sum(p.numel() for p in denoiser.parameters()),
    )
    results = pd.DataFrame([vae_metrics, gan_metrics, ddpm_metrics])
    results.to_csv(OUT / "results.csv", index=False)
    pd.DataFrame(gan_checkpoints).to_csv(OUT / "gan_checkpoint_histograms.csv", index=False)
    pd.DataFrame(evaluator_history).to_csv(OUT / "evaluator_history.csv", index=False)
    pd.DataFrame(vae_history).to_csv(OUT / "vae_history.csv", index=False)
    pd.DataFrame(gan_history).to_csv(OUT / "gan_history.csv", index=False)
    pd.DataFrame(diffusion_history).to_csv(OUT / "diffusion_history.csv", index=False)

    history_fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    evaluator_frame = pd.DataFrame(evaluator_history)
    axes[0, 0].plot(evaluator_frame["epoch"], evaluator_frame["loss"], label="loss")
    axes[0, 0].set_title("Evaluator loss")
    vae_frame = pd.DataFrame(vae_history)
    axes[0, 1].plot(vae_frame["epoch"], vae_frame["recon_loss"], label="reconstruction")
    axes[0, 1].plot(vae_frame["epoch"], vae_frame["kl"], label="KL")
    axes[0, 1].plot(vae_frame["epoch"], vae_frame["total_loss"], label="total")
    axes[0, 1].set_title("VAE losses")
    axes[0, 1].legend()
    gan_frame = pd.DataFrame(gan_history)
    axes[1, 0].plot(gan_frame["epoch"], gan_frame["g_loss"], label="generator")
    axes[1, 0].plot(gan_frame["epoch"], gan_frame["d_loss"], label="discriminator")
    axes[1, 0].set_title("GAN losses")
    axes[1, 0].legend()
    diffusion_frame = pd.DataFrame(diffusion_history)
    axes[1, 1].plot(diffusion_frame["step"], diffusion_frame["mse"], marker="o")
    axes[1, 1].set_title("Diffusion noise-prediction MSE")
    for axis in axes.flat:
        axis.grid(alpha=0.25)
    history_fig.tight_layout()
    history_fig.savefig(OUT / "training_curves.png", dpi=160)
    plt.close(history_fig)

    evaluator_fig, evaluator_axis = plt.subplots(figsize=(8, 5))
    evaluator_axis.plot(evaluator_frame["epoch"], evaluator_frame["loss"], marker="o", label="cross-entropy loss")
    evaluator_axis.set_xlabel("Epoch")
    evaluator_axis.set_ylabel("Cross-entropy loss")
    evaluator_axis.set_title("Evaluator training loss")
    evaluator_axis.grid(alpha=0.25)
    evaluator_axis.legend()
    evaluator_fig.tight_layout()
    evaluator_fig.savefig(OUT / "evaluator_training_results.png", dpi=160)
    plt.close(evaluator_fig)

    print(results)

    checkpoint_frame = pd.DataFrame(gan_checkpoints).set_index("epoch")
    axis = checkpoint_frame.plot(kind="bar", stacked=True, figsize=(10, 5), ylim=(0, 1), title="GAN evaluator-label histogram by checkpoint")
    axis.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(rect=(0, 0, 0.82, 1))
    plt.savefig(OUT / "gan_mode_collapse_checkpoints.png", dpi=160)
    plt.close()

    diffusion_speed_rows = []
    for sampling_steps in (25, 100, CONFIG["diffusion_steps"]):
        synchronize_cuda()
        started = __import__("time").perf_counter()
        diffusion_samples = sample_diffusion(diffusion, denoiser, CONFIG["metric_count"], steps=sampling_steps)
        synchronize_cuda()
        elapsed = __import__("time").perf_counter() - started
        labels, features = evaluator_outputs(diffusion_samples, evaluator)
        real_features, real_labels = reference_features(CONFIG["metric_count"], evaluator)
        speed_metrics = coverage_metrics(real_features, real_labels, features.numpy(), labels.numpy())
        speed_metrics.update({
            "sampling_steps": sampling_steps,
            "images_per_second": CONFIG["metric_count"] / elapsed,
            "time_for_5000_images": 5000 * elapsed / CONFIG["metric_count"],
        })
        diffusion_speed_rows.append(speed_metrics)
        show_grid(diffusion_samples[:CONFIG["sample_count"]], f"DDPM samples ({sampling_steps} steps)", f"diffusion_samples_{sampling_steps}_steps.png")
    pd.DataFrame(diffusion_speed_rows).drop(columns="generated_histogram").to_csv(OUT / "diffusion_speed_quality.csv", index=False)

    preview, _ = next(iter(test_loader))
    preview = preview[:10].to(DEVICE)
    with torch.no_grad():
        reconstruction, _, _ = vae(preview)
        mu0, _ = vae.encode(preview[0:1])
        mu1, _ = vae.encode(preview[1:2])
        vae_interp = torch.cat([
            vae.decode(torch.lerp(mu0, mu1, f))
            for f in torch.linspace(0, 1, 10, device=DEVICE)
        ])
        z0, z1 = torch.randn(1, generator.latent_dim, device=DEVICE), torch.randn(1, generator.latent_dim, device=DEVICE)
        gan_interp = torch.cat([generator(torch.lerp(z0, z1, f)) for f in torch.linspace(0, 1, 10, device=DEVICE)]).cpu()
    show_grid(torch.cat([preview.cpu(), reconstruction.cpu()]), "VAE originals and reconstructions", "vae_reconstruction.png")
    show_grid(vae_interp.cpu(), "VAE interpolation between two real test images", "vae_interpolation.png", columns=10)
    show_grid(gan_interp, "GAN latent interpolation", "gan_interpolation.png", columns=10)
    show_grid(sample_vae(vae, CONFIG["sample_count"]), "VAE prior samples", "vae_samples.png")
    show_grid(sample_gan(generator, CONFIG["sample_count"]), "GAN samples", "gan_samples.png")
    show_grid(sample_diffusion(diffusion, denoiser, CONFIG["sample_count"], steps=min(CONFIG["diffusion_steps"], 100)), "DDPM samples", "diffusion_samples.png")
    print("Saved artifacts to:", OUT)




In [ ]:
# Execute the complete experiment before displaying the saved results.
run_experiment()


## Results and Required Figures

After the run, this cell displays the metrics table and saved figures in the same order as the assignment.

In [ ]:
from IPython.display import Image, display
results_frame = pd.read_csv(OUT / "results.csv")
display(results_frame)
for filename in [
    "evaluator_training_results.png", "vae_reconstruction.png", "vae_samples.png",
    "vae_interpolation.png", "gan_samples.png", "gan_interpolation.png",
    "gan_mode_collapse_checkpoints.png", "diffusion_samples_25_steps.png",
    "diffusion_samples_100_steps.png", "diffusion_samples_500_steps.png",
    "training_curves.png",
]:
    path = OUT / filename
    if path.exists():
        display(Image(filename=str(path)))


## Submission Notes

The notebook is self-contained; no Python source modules are required for execution. The generated CSV files and figures are saved under `assignment1_outputs`. Save this notebook after execution so the training logs, metrics table, and figures remain visible to the grader.